# Module 1 · Chapter 2 — 퍼짐을 측정하다: 분산·표준편차·IQR

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 시나리오·Why·Where·직관·정의·수식·함정·다음 질문
- **이 노트북**: 피트니스 센터 두 트레이너 데이터로 퍼짐 측정 도구를 직접 계산

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 시나리오 데이터

두 트레이너(A, B) 고객 각 10명의 주간 운동 시간(시간)을 만듭니다.
- **평균은 둘 다 5.0시간** 으로 같습니다.
- **A 트레이너**: 고르게 분포 (분산 작음)
- **B 트레이너**: 들쭉날쭉 (분산 큼)

In [ ]:
trainer_a = np.array([4.2, 4.5, 4.7, 4.8, 5.0, 5.0, 5.1, 5.3, 5.5, 5.9])
trainer_b = np.array([1.0, 2.0, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.0, 9.0])

print(f"A 평균: {trainer_a.mean():.2f}h  B 평균: {trainer_b.mean():.2f}h")
print(f"A 합계: {trainer_a.sum():.1f}    B 합계: {trainer_b.sum():.1f}")

df = pd.DataFrame({
    "운동시간": np.concatenate([trainer_a, trainer_b]),
    "트레이너": ["A"] * 10 + ["B"] * 10,
})

## 2. 직관 — 편차(deviation) 시각화

각 데이터 포인트가 평균에서 얼마나 떨어져 있는지를 화살표로 봅니다.
그리고 편차를 모두 더하면 왜 0이 되는지 직접 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, data, label, color in zip(
    axes,
    [trainer_a, trainer_b],
    ["A 트레이너", "B 트레이너"],
    ["steelblue", "tomato"],
):
    mean = data.mean()
    ax.axhline(mean, color="k", linewidth=1.5, linestyle="--", label=f"평균 = {mean:.1f}")
    for i, x in enumerate(data):
        ax.plot(i, x, "o", color=color, markersize=8)
        ax.vlines(i, mean, x, colors=color, alpha=0.5, linewidth=2)
    ax.set_title(label, fontsize=13)
    ax.set_xlabel("고객 번호")
    ax.set_ylabel("운동 시간 (h)")
    ax.legend()

plt.suptitle("편차(deviation): 각 점에서 평균까지의 거리", fontsize=14)
plt.tight_layout()
plt.show()

# 편차의 합은 항상 0
dev_a = trainer_a - trainer_a.mean()
dev_b = trainer_b - trainer_b.mean()
print(f"A 편차 합: {dev_a.sum():.10f}  (= 0 이어야 함)")
print(f"B 편차 합: {dev_b.sum():.10f}  (= 0 이어야 함)")
print("→ 편차를 그냥 더하면 퍼짐을 측정할 수 없습니다. 그래서 '제곱'합니다.")

## 3. 손으로 계산해 보기

수식을 코드로 한 줄씩 옮깁니다.

$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

In [ ]:
def manual_stats(data, label):
    n = len(data)
    mean = sum(data) / n                          # 평균
    deviations = data - mean                      # 편차
    sq_dev = deviations ** 2                      # 편차 제곱
    variance = sum(sq_dev) / (n - 1)              # 표본 분산 (n-1)
    std = variance ** 0.5                         # 표준편차

    sorted_data = np.sort(data)
    q1 = np.percentile(sorted_data, 25)
    q3 = np.percentile(sorted_data, 75)
    iqr = q3 - q1

    print(f"\n[{label}]")
    print(f"  분산(s²)     = {variance:.4f}")
    print(f"  표준편차(s)  = {std:.4f}")
    print(f"  Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.4f}")
    return variance, std, iqr

var_a, std_a, iqr_a = manual_stats(trainer_a, "A 트레이너")
var_b, std_b, iqr_b = manual_stats(trainer_b, "B 트레이너")

## 4. 라이브러리로 같은 답 재현

NumPy와 Pandas로 같은 결과를 내는지 확인합니다.

**주의**: `np.var()` 와 `pd.Series.var()` 의 기본 `ddof` 값이 다릅니다.

In [ ]:
print("=== 분산 비교 ===")
print(f"np.var(A, ddof=1)    = {np.var(trainer_a, ddof=1):.4f}  ← 표본 분산 (n-1)")
print(f"np.var(A, ddof=0)    = {np.var(trainer_a, ddof=0):.4f}  ← 모분산 (n), NumPy 기본값")
print(f"pd.Series(A).var()   = {pd.Series(trainer_a).var():.4f}  ← 표본 분산, Pandas 기본값")
print()

print("=== 표준편차 비교 ===")
print(f"np.std(A, ddof=1)    = {np.std(trainer_a, ddof=1):.4f}  ← 표본 표준편차")
print(f"np.std(A, ddof=0)    = {np.std(trainer_a, ddof=0):.4f}  ← 모표준편차")
print(f"pd.Series(A).std()   = {pd.Series(trainer_a).std():.4f}  ← 표본 표준편차")
print()

print("=== IQR ===")
from scipy import stats
print(f"scipy.stats.iqr(A)  = {stats.iqr(trainer_a):.4f}")
print(f"직접 계산           = {iqr_a:.4f}")

## 5. 함정 점검 — n vs n-1

표본 크기가 작을수록 `ddof=0` 과 `ddof=1` 의 차이가 커집니다.
실제로 얼마나 차이가 나는지 확인합니다.

In [ ]:
sample_sizes = [5, 10, 20, 50, 100, 500]
true_std = 2.0  # 알고 있다고 가정한 모표준편차

results = []
for n in sample_sizes:
    samples = [rng.normal(0, true_std, n) for _ in range(1000)]
    biased   = np.mean([np.std(s, ddof=0) for s in samples])  # ddof=0
    unbiased = np.mean([np.std(s, ddof=1) for s in samples])  # ddof=1
    results.append({"n": n, "ddof=0": biased, "ddof=1": unbiased})

result_df = pd.DataFrame(results)
print(f"실제 모표준편차 = {true_std:.1f}")
print()
print(result_df.to_string(index=False, float_format="{:.4f}".format))
print()
print("→ n이 작을수록 ddof=0은 모표준편차를 과소 추정합니다 (편향).")
print("→ ddof=1(표본 표준편차)이 모표준편차의 불편(unbiased) 추정값입니다.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 분포 비교
axes[0].hist(trainer_a, bins=6, alpha=0.6, label="A 트레이너", color="steelblue")
axes[0].hist(trainer_b, bins=6, alpha=0.6, label="B 트레이너", color="tomato")
axes[0].axvline(trainer_a.mean(), color="steelblue", linestyle="--", linewidth=2)
axes[0].axvline(trainer_b.mean(), color="tomato", linestyle="--", linewidth=2)
axes[0].set_title("같은 평균, 다른 분산")
axes[0].set_xlabel("운동 시간 (h)")
axes[0].legend()

# 표준편차 비교 바 차트
axes[1].bar(["A 표준편차", "B 표준편차"], [std_a, std_b],
            color=["steelblue", "tomato"], alpha=0.8)
axes[1].set_ylabel("표준편차 (h)")
axes[1].set_title("표준편차 크기 비교")
for i, v in enumerate([std_a, std_b]):
    axes[1].text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

## 6. 직접 답해 보기

1. 표본 분산의 분모가 $n-1$ 인 이유를 자신의 말로 설명한다면?
2. 두 그룹의 평균이 같을 때, 어떤 추가 정보가 있어야 "두 그룹이 같다"고 말할 수 있을까?
3. 분산 대신 IQR을 쓰는 것이 더 나은 상황은 언제인가?